# 119. 현재시점 이전 비라벨100개를 이용한 국소 공정 교사
116번의 전역 관계 잔차 구간변동을 본 뒤 만든 개발가설. 모든 원래라벨 Timestamp를 교사참조에서 제거. 각현재샷 이전 비라벨최대100개만 사용하고30분공백에서초기화, 최소20개. 미래센서/정답을 교사에 쓰지 않는다.
116번과 달리 평가구간 안의 이전 비라벨 센서도 온라인특징의 이력으로 사용할 수 있다. 따라서 평가구간 센서 전부를 배제하는 프로토콜과 같지 않다. 품질분류모델·대체값·앙상블·임계값에서는 여전히 평가구간 라벨 전체 제외. 이력이 주어지는 온라인 배포 가정이고 공식테스트 제공여부 미확인. 0.5미달을 성공으로 바꾸지 않음.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
labels=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=labels[labels['product'].eq('RG3')&(labels.PassOrFail.eq(0)|labels.Reason.str.contains('미성형',na=False))].reset_index(drop=True)
y=d.PassOrFail.to_numpy(int);g=d.comparison_block.to_numpy();blocks=np.unique(g)
assert len(d)==574 and y.sum()==8
u=pd.read_csv(root/'data/original_split/moldset_unlabeled_rg3.csv',parse_dates=['TimeStamp'])
u=u[u.EQUIP_CD.isin(d.EQUIP_CD.unique())].sort_values('TimeStamp').drop_duplicates(['EQUIP_CD','TimeStamp'])
u=u[~u.TimeStamp.isin(labels.TimeStamp)].reset_index(drop=True)
relations={
 'pressure':('Max_Injection_Pressure',['Max_Switch_Over_Pressure','Max_Injection_Speed','Filling_Time']),
 'fill':('Filling_Time',['Max_Injection_Pressure','Max_Injection_Speed','Mold_Temperature_3']),
 'cushion':('Cushion_Position',['Plasticizing_Position','Average_Back_Pressure','Barrel_Temperature_5']),
 'recovery':('Plasticizing_Time',['Max_Screw_RPM','Plasticizing_Position','Average_Back_Pressure'])}
# Online feature construction uses only past unlabeled sensor observations.
# Past unlabeled rows inside an evaluation interval can be used; no quality labels.
ut=u.TimeStamp.to_numpy(); features=np.full((len(d),4),np.nan);coverage=[]
for i,row in d.iterrows():
 stop=np.searchsorted(ut,row.TimeStamp.to_datetime64(),side='left')
 past=u.iloc[max(0,stop-100):stop].copy()
 if len(past):
  gap=past.TimeStamp.diff().dt.total_seconds()
  split=np.flatnonzero(gap.gt(1800).to_numpy())
  if len(split):past=past.iloc[split[-1]:]
 usable=len(past)>=20 and (row.TimeStamp-past.TimeStamp.iloc[-1]).total_seconds()<=1800
 coverage.append(dict(TimeStamp=row.TimeStamp,actual=int(y[i]),prior_unlabeled=len(past),usable=int(usable)))
 if not usable:continue
 assert past.TimeStamp.max()<row.TimeStamp
 for j,(name,(target,parents)) in enumerate(relations.items()):
  a=past[parents+[target]].dropna()
  if len(a)<20:continue
  sc=RobustScaler().fit(a[parents]);xp=sc.transform(a[parents])
  m=Ridge(alpha=1).fit(xp,a[target])
  residual=a[target].to_numpy()-m.predict(xp)
  scale=max(np.std(residual),float(a[target].std())*.1,1e-6)
  q=pd.DataFrame([row[parents].to_numpy(float)],columns=parents)
  features[i,j]=(float(row[target])-float(m.predict(sc.transform(q))[0]))/scale
print('Causal coverage',sum(z['usable'] for z in coverage),'of574; all8',sum(z['usable'] for z in coverage if z['actual']),flush=True)
views={'local_residual4':[0,1,2,3],'local_pressure2':[0,1],'local_supply2':[2,3]}
names=list(views);memo={}
def predict(ref,q):
 assert set(g[ref]).isdisjoint(g[q])
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(features[ref[y[ref]==0]])
 a=np.clip(imp.transform(features),-12,12);out=[]
 for name,cols in views.items():
  scores=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2)
    scores.append(m.fit(a[ref][:,cols],y[ref]).predict_proba(a[q][:,cols])[:,1])
  out.append(np.mean(scores,axis=0))
 result=np.column_stack(out);memo[key]=result;return result
weights=[(n,np.eye(3)[j]) for j,n in enumerate(names)]+[('local_expert_mean',np.ones(3)/3)]
def threshold(yy,s):
 p,r,c=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)
 k=max(range(len(c)),key=lambda k:(f[k],r[k],c[k]));return float(c[k]),float(f[k])
records=[];choices=[]
for b in blocks:
 ref=np.flatnonzero(g!=b);q=np.flatnonzero(g==b);out=predict(ref,q)
 inner=np.full((len(ref),3),np.nan)
 for ib in np.unique(g[ref]):
  mask=g[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for name,w in weights:
  cal=inner@w;score=out@w;cut,fi=threshold(y[ref],cal);candidates.append((fi,name,w,cut))
  for rule,th in [('inner_F1',cut),('normal99',float(np.quantile(cal[y[ref]==0],.99,method='higher')))]:
   pr=score>=th
   for j,i in enumerate(q):records.append(dict(model=name,rule=rule,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[j]),threshold=th,prediction=int(pr[j])))
 fi,name,w,th=max(candidates,key=lambda z:(z[0],-np.count_nonzero(z[2]),z[1]));score=out@w;pr=score>=th
 choices.append(dict(block=b,selected=name,threshold=th,inner_F1=fi))
 for j,i in enumerate(q):records.append(dict(model='primary_inner_selected',rule='inner_F1',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[j]),threshold=th,prediction=int(pr[j])))
 print('Completed causal teacher',b,name,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,rule),z in p.groupby(['model','rule']):
 assert len(z)==574 and z.TimeStamp.is_unique
 tn,fp,fn,tp=confusion_matrix(z.actual,z.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,rule=rule,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(z.actual,z.score)))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(coverage),'coverage'),(pd.DataFrame(choices),'choices')]:
 obj.to_csv(t/('rg3_shortshot_causal_teacher_'+name+'.csv'),index=False,encoding='utf-8-sig')
pd.DataFrame(features,columns=list(relations)).assign(TimeStamp=d.TimeStamp,actual=y).to_csv(t/'rg3_shortshot_causal_teacher_residuals.csv',index=False,encoding='utf-8-sig')
print(m.sort_values('F1',ascending=False).to_string(index=False))


Causal coverage 111 of574; all8 1


Completed causal teacher 20201021_001626_RG3 local_residual4


Completed causal teacher 20201022_002405_RG3 local_residual4


Completed causal teacher 20201022_044026_RG3 local_supply2


Completed causal teacher 20201023_010128_RG3 local_supply2


Completed causal teacher 20201023_044148_RG3 local_supply2


                 model     rule  TP  FP  FN  TN       F1       AP
     local_expert_mean inner_F1   3 162   5 404 0.034682 0.016576
       local_residual4 inner_F1   3 162   5 404 0.034682 0.016560
primary_inner_selected inner_F1   3 162   5 404 0.034682 0.016701
         local_supply2 inner_F1   3 163   5 403 0.034483 0.016687
       local_pressure2 inner_F1   3 166   5 400 0.033898 0.016251
       local_pressure2 normal99   0   4   8 562 0.000000 0.016251
     local_expert_mean normal99   0   0   8 566 0.000000 0.016576
       local_residual4 normal99   0   0   8 566 0.000000 0.016560
         local_supply2 normal99   0   1   8 565 0.000000 0.016687
